# ML-09 — Validation and Research Claim Audit

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/thar-26/flyrank-ml-internship/blob/main/work/notebooks/w06_validation_audit.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

In [4]:
import os
from pathlib import Path
import pandas as pd
import numpy as np

REPO_DIR = Path("/content/flyrank-ml-internship")

if not REPO_DIR.exists():
    !git clone https://github.com/thar-26/flyrank-ml-internship.git /content/flyrank-ml-internship

%cd /content/flyrank-ml-internship

DATA_PATH = Path("data/raw/content_refresh_anonymized.csv")

assert DATA_PATH.exists(), f"Dataset not found: {DATA_PATH}"

df = pd.read_csv(DATA_PATH)

# Same evaluation proxy used in earlier weeks.
df["is_declining_label"] = (
    df["trend_direction"] == "down"
).astype(int)

print("Current directory:", Path.cwd())
print("Rows:", len(df))
print("Clients:", df["client_id"].nunique())
print("Proxy target rate:", round(df["is_declining_label"].mean(), 3))
print("PASS: Dataset loaded and evaluation proxy created.")

/content/flyrank-ml-internship
Current directory: /content/flyrank-ml-internship
Rows: 30000
Clients: 32
Proxy target rate: 0.542
PASS: Dataset loaded and evaluation proxy created.


## 1. Two paper findings + my methodology questions

*Pick two findings from the FlyRank research paper. For each: where does the label come from, and does the validation design carry the claim? Constructive tone.*


## 1) Two paper findings + my methodology questions

### Finding 1: Statistical significance alone does not show whether a result is practically meaningful

The paper explains that a statistically significant P value only provides evidence about whether an observed difference is unlikely to be due to chance. It does not describe the magnitude or practical importance of that difference. With a sufficiently large sample, even a very small effect can become statistically significant.

**My methodology question:**  
When a study reports a statistically significant result, does it also report an appropriate effect size and enough context to determine whether the observed difference is practically meaningful?

This is important because statistical significance alone may support a stronger conclusion than the actual magnitude of the observed effect justifies.


### Finding 2: Statistical power depends on both effect size and sample size

The paper explains that smaller effects generally require larger samples to be detected reliably, while larger effects may be detected with smaller samples. It also recommends estimating effect size and statistical power before conducting a study when planning the required sample size.

**My methodology question:**  
Was the sample size planned using a justified expected effect size and power analysis, and were those assumptions based on pilot data, previous research, or a clearly defined practically meaningful difference?

This matters because an underpowered study may fail to detect a meaningful effect, while a very large sample may make a trivial effect appear statistically significant.


### Reflection

These findings are relevant to my own machine learning work because reporting only a model metric does not show whether the improvement is meaningful or robust. I should compare my model against the same baseline, on the same evaluation data, report the size of the observed improvement, and use an honest validation design before making claims about model performance.

In [2]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 2. My model under an honest split (before/after)

*Re-run your Week-5 model under a grouped or time-aware split. Show both numbers.*

## 2) My model under an honest split (before/after)

In Week 5, I evaluated the model using a grouped train/test split by client. This is the more realistic validation design because content pages from the same client may share patterns that a model could learn.

For this audit, I also reproduce a random row split as the weaker "before" design. A random split can place pages from the same client in both training and test data, which may produce an optimistic estimate of performance.

I compare the same Random Forest model, feature set, random seed, and evaluation metrics under both validation designs. The only intended change is the split strategy.

The grouped split is treated as the improved validation design because it measures performance on clients that were not seen during training.

In [5]:
from sklearn.model_selection import train_test_split, GroupShuffleSplit
from sklearn.ensemble import RandomForestClassifier

SEED = 42

FEATURES = [
    "impressions_90d",
    "days_since_last_update",
    "content_age_days",
    "avg_position",
    "ctr",
    "engagement_rate",
]

TARGET = "is_declining_label"
GROUP = "client_id"


def precision_at_k(y_true, scores, k):
    y_true = np.asarray(y_true)
    scores = np.asarray(scores)

    order = np.argsort(-scores)

    return y_true[order[:k]].mean()


# --------------------------------------------------
# DATA
# --------------------------------------------------

X = df[FEATURES].copy()
y = df[TARGET].copy()


# ==================================================
# BEFORE: RANDOM ROW SPLIT
# ==================================================

X_train_random, X_test_random, y_train_random, y_test_random = (
    train_test_split(
        X,
        y,
        test_size=0.25,
        random_state=SEED,
        stratify=y,
    )
)

rf_random = RandomForestClassifier(
    n_estimators=300,
    min_samples_leaf=10,
    random_state=SEED,
    n_jobs=-1,
)

rf_random.fit(
    X_train_random,
    y_train_random,
)

random_scores = rf_random.predict_proba(
    X_test_random
)[:, 1]

random_p20 = precision_at_k(
    y_test_random.to_numpy(),
    random_scores,
    20,
)

random_p50 = precision_at_k(
    y_test_random.to_numpy(),
    random_scores,
    50,
)


# ==================================================
# AFTER: GROUPED SPLIT BY CLIENT
# ==================================================

gss = GroupShuffleSplit(
    n_splits=1,
    test_size=0.25,
    random_state=SEED,
)

train_idx, test_idx = next(
    gss.split(
        df[FEATURES],
        df[TARGET],
        groups=df[GROUP],
    )
)

train_grouped = df.iloc[train_idx].copy()
test_grouped = df.iloc[test_idx].copy()

X_train_grouped = train_grouped[FEATURES]
y_train_grouped = train_grouped[TARGET]

X_test_grouped = test_grouped[FEATURES]
y_test_grouped = test_grouped[TARGET]

rf_grouped = RandomForestClassifier(
    n_estimators=300,
    min_samples_leaf=10,
    random_state=SEED,
    n_jobs=-1,
)

rf_grouped.fit(
    X_train_grouped,
    y_train_grouped,
)

grouped_scores = rf_grouped.predict_proba(
    X_test_grouped
)[:, 1]

grouped_p20 = precision_at_k(
    y_test_grouped.to_numpy(),
    grouped_scores,
    20,
)

grouped_p50 = precision_at_k(
    y_test_grouped.to_numpy(),
    grouped_scores,
    50,
)


# ==================================================
# GROUPED VALIDATION CHECK
# ==================================================

train_clients = set(train_grouped[GROUP])
test_clients = set(test_grouped[GROUP])

client_overlap = len(
    train_clients.intersection(test_clients)
)

assert client_overlap == 0


# ==================================================
# COMPARISON TABLE
# ==================================================

comparison = pd.DataFrame(
    {
        "Validation design": [
            "Random row split (before)",
            "Grouped client split (after)",
        ],
        "Test rows": [
            len(y_test_random),
            len(y_test_grouped),
        ],
        "Test base rate": [
            y_test_random.mean(),
            y_test_grouped.mean(),
        ],
        "Precision@20": [
            random_p20,
            grouped_p20,
        ],
        "Precision@50": [
            random_p50,
            grouped_p50,
        ],
    }
)


# ==================================================
# OUTPUT
# ==================================================

print("VALIDATION AUDIT — BEFORE VS AFTER")
print()

print("MODEL CONFIGURATION")
print("Random Forest estimators: 300")
print("Random Forest min_samples_leaf: 10")
print("Random seed:", SEED)

print()

print("Random split test rows:", len(y_test_random))
print("Grouped split test rows:", len(y_test_grouped))

print()

print("Grouped training clients:", len(train_clients))
print("Grouped test clients:", len(test_clients))
print("Client overlap:", client_overlap)

print()
print("PASS: No client appears in both grouped train and test sets.")

print()
print("VALIDATION COMPARISON")

display(comparison.round(3))


# ==================================================
# OBSERVED VALIDATION GAP
# ==================================================

p20_gap = random_p20 - grouped_p20
p50_gap = random_p50 - grouped_p50

print()
print("OBSERVED VALIDATION GAP")

print(
    f"Precision@20 difference "
    f"(random - grouped): {p20_gap:.3f}"
)

print(
    f"Precision@50 difference "
    f"(random - grouped): {p50_gap:.3f}"
)

if p20_gap > 0 and p50_gap > 0:

    print(
        "Interpretation: The random row split produced higher measured "
        "ranking metrics at both review depths."
    )

elif p20_gap < 0 and p50_gap < 0:

    print(
        "Interpretation: The grouped client split produced higher measured "
        "ranking metrics at both review depths."
    )

else:

    print(
        "Interpretation: The effect of validation design is mixed across "
        "review depths. The grouped split achieved higher Precision@20, "
        "while the random row split achieved higher Precision@50. "
        "The grouped design remains preferable because it evaluates "
        "performance on unseen clients."
    )

VALIDATION AUDIT — BEFORE VS AFTER

MODEL CONFIGURATION
Random Forest estimators: 300
Random Forest min_samples_leaf: 10
Random seed: 42

Random split test rows: 7500
Grouped split test rows: 7115

Grouped training clients: 24
Grouped test clients: 8
Client overlap: 0

PASS: No client appears in both grouped train and test sets.

VALIDATION COMPARISON


,Validation design,Test rows,Test base rate,Precision@20,Precision@50
0,Random row split (before),7500,0.542,0.8,0.86
1,Grouped client split (after),7115,0.517,0.9,0.70



OBSERVED VALIDATION GAP
Precision@20 difference (random - grouped): -0.100
Precision@50 difference (random - grouped): 0.160
Interpretation: The effect of validation design is mixed across review depths. The grouped split achieved higher Precision@20, while the random row split achieved higher Precision@50. The grouped design remains preferable because it evaluates performance on unseen clients.


## 3. Leakage audit

*The same hunt from Week 3, on your final feature set.*

## 3) Leakage audit

I audited the model inputs against three leakage risks: label-derived features, future or overlapping-window features, and decision-derived product flags.

The proxy label is created from trend_direction, so trend_direction and its sibling trend_pct are excluded from the model features. The model also excludes identifiers such as client_id and content_id, and it does not use existing product flags, scores, or recommendation outputs as features.

The six model inputs are current page-level signals. However, the dataset does not provide enough timestamp metadata to prove that every feature window is strictly earlier than the proxy-label window. Therefore, I treat timeline separation as an unresolved validation limitation rather than claiming that temporal leakage has been fully ruled out.

The grouped validation design keeps every client entirely in either training or test data. This prevents direct client overlap and provides a more conservative estimate of performance on unseen clients.

In [6]:
# --------------------------------------------------
# LEAKAGE AUDIT
# --------------------------------------------------

model_features = set(FEATURES)

label_derived_columns = {
    "trend_direction",
    "trend_pct",
    "is_declining_label",
}

identifier_columns = {
    "client_id",
    "content_id",
}

decision_derived_columns = {
    col
    for col in df.columns
    if (
        "flag" in col.lower()
        or "score" in col.lower()
        or "recommend" in col.lower()
    )
}

label_leaks = model_features.intersection(
    label_derived_columns
)

identifier_leaks = model_features.intersection(
    identifier_columns
)

decision_leaks = model_features.intersection(
    decision_derived_columns
)


print("LEAKAGE AUDIT")
print()

print("Model features:")

for feature in FEATURES:
    print("-", feature)

print()

print(
    "Label-derived inputs used:",
    sorted(label_leaks),
)

print(
    "Identifier inputs used:",
    sorted(identifier_leaks),
)

print(
    "Decision-derived inputs used:",
    sorted(decision_leaks),
)


print()
print("TIMELINE CHECK")

print(
    "LIMITATION: The available dataset does not provide enough "
    "timestamp metadata to verify that every feature window is "
    "strictly earlier than the proxy-label window."
)


# --------------------------------------------------
# ASSERTIONS
# --------------------------------------------------

assert len(label_leaks) == 0
assert len(identifier_leaks) == 0
assert len(decision_leaks) == 0
assert client_overlap == 0


print()

print(
    "PASS: No label-derived columns are used as model features."
)

print(
    "PASS: No identifiers are used as model features."
)

print(
    "PASS: No detected product flags, scores, or recommendation "
    "outputs are used."
)

print(
    "PASS: No client overlap exists in the grouped split."
)

print(
    "CAUTION: Temporal leakage cannot be fully ruled out "
    "from the available metadata."
)

LEAKAGE AUDIT

Model features:
- impressions_90d
- days_since_last_update
- content_age_days
- avg_position
- ctr
- engagement_rate

Label-derived inputs used: []
Identifier inputs used: []
Decision-derived inputs used: []

TIMELINE CHECK
LIMITATION: The available dataset does not provide enough timestamp metadata to verify that every feature window is strictly earlier than the proxy-label window.

PASS: No label-derived columns are used as model features.
PASS: No identifiers are used as model features.
PASS: No detected product flags, scores, or recommendation outputs are used.
PASS: No client overlap exists in the grouped split.
CAUTION: Temporal leakage cannot be fully ruled out from the available metadata.


## 4. Claim rewrite

*Take your own boldest sentence and rewrite it in safe language: observed, measured, directional, decision-support.*

## 4) Claim rewrite

### Week-5 claim before the validation audit

Random Forest provides the strongest top-of-queue ranking and improves refresh prioritization compared with the simpler alternatives.

### Rewritten claim after the validation audit

On one client-grouped holdout split, the Random Forest achieved a measured Precision@20 of 0.900 and Precision@50 of 0.700. Compared with a random row split using the same features, model configuration, seed, and evaluation metrics, the grouped split produced higher Precision@20 but lower Precision@50.

The effect of validation design was therefore mixed across review depths. The 0.160 decrease in Precision@50 under client-grouped validation suggests that the random row split gave a more optimistic estimate deeper in the ranked queue. However, Precision@20 increased by 0.100 under grouped validation, so the audit does not support a general claim that random-row validation inflated every ranking metric.

The grouped split is still the preferred validation design for this task because it measures performance on clients that were not present during training. The observed results provide directional evidence that the model can support refresh-prioritization decisions, but they do not establish performance across all unseen clients or future time periods.

The leakage audit found no label-derived features, identifiers, or detected product flags, scores, or recommendation outputs in the model inputs. Client overlap was also zero under grouped validation. However, the available dataset metadata was not sufficient to verify strict temporal separation between all feature windows and the proxy-label window, so temporal leakage cannot be fully ruled out.

Overall, I would describe the model as a decision-support ranking approach that showed useful measured performance on one client-grouped holdout. Stronger claims about generalization would require repeated grouped validation or GroupKFold evaluation, and stronger deployment claims would require a time-aware validation design with clearly separated feature and outcome windows.

In [7]:
# --------------------------------------------------
# FAILURE EXAMPLES FROM HONEST GROUPED HOLDOUT
# --------------------------------------------------

grouped_queue = test_grouped[
    [
        "content_id",
        "impressions_90d",
        "days_since_last_update",
        "content_age_days",
        "avg_position",
        "ctr",
        "engagement_rate",
        "is_declining_label",
    ]
].copy()

grouped_queue["model_score"] = grouped_scores

grouped_queue = (
    grouped_queue
    .sort_values(
        "model_score",
        ascending=False,
    )
    .reset_index(drop=True)
)

grouped_queue["rank"] = (
    np.arange(len(grouped_queue)) + 1
)


# --------------------------------------------------
# FALSE POSITIVES IN TOP 50
# --------------------------------------------------

top50_false_positives = grouped_queue[
    (grouped_queue["rank"] <= 50)
    &
    (grouped_queue["is_declining_label"] == 0)
].copy()

wrong_cases = (
    top50_false_positives
    .head(3)
    .copy()
)


# --------------------------------------------------
# OUTPUT
# --------------------------------------------------

print("FAILURE EXAMPLES — GROUPED HOLDOUT")
print()

print(
    "False positives in top 50:",
    len(top50_false_positives),
)

print()

display(
    wrong_cases[
        [
            "rank",
            "content_id",
            "model_score",
            "impressions_90d",
            "days_since_last_update",
            "content_age_days",
            "avg_position",
            "ctr",
            "engagement_rate",
            "is_declining_label",
        ]
    ].round(3)
)


print()
print("WHY THESE CASES ARE HARD")


for _, row in wrong_cases.iterrows():

    print(
        f"Rank {int(row['rank'])}: "
        f"The model assigned a high score of "
        f"{row['model_score']:.3f}, "
        f"but the evaluation proxy marks the page "
        f"as non-declining. "
        f"The page has "
        f"{int(row['impressions_90d']):,} impressions, "
        f"{int(row['days_since_last_update'])} days "
        f"since its last update, "
        f"average position {row['avg_position']:.1f}, "
        f"CTR {row['ctr']:.2f}, "
        f"and engagement rate "
        f"{row['engagement_rate']:.2f}. "
        f"This observed feature pattern resembles "
        f"declining pages learned from training clients, "
        f"but it does not match the proxy outcome "
        f"for this held-out page."
    )

FAILURE EXAMPLES — GROUPED HOLDOUT

False positives in top 50: 15



,rank,content_id,model_score,impressions_90d,days_since_last_update,content_age_days,avg_position,ctr,engagement_rate,is_declining_label
1,2,content_91c02d2830b8,0.952,720,20,147,19.4,0.0,0.0,0
15,16,content_68742c70d0fa,0.921,660,20,147,7.4,0.0,0.0,0
20,21,content_fb79d33c4d8a,0.920,978,20,147,23.9,0.0,0.0,0



WHY THESE CASES ARE HARD
Rank 2: The model assigned a high score of 0.952, but the evaluation proxy marks the page as non-declining. The page has 720 impressions, 20 days since its last update, average position 19.4, CTR 0.00, and engagement rate 0.00. This observed feature pattern resembles declining pages learned from training clients, but it does not match the proxy outcome for this held-out page.
Rank 16: The model assigned a high score of 0.921, but the evaluation proxy marks the page as non-declining. The page has 660 impressions, 20 days since its last update, average position 7.4, CTR 0.00, and engagement rate 0.00. This observed feature pattern resembles declining pages learned from training clients, but it does not match the proxy outcome for this held-out page.
Rank 21: The model assigned a high score of 0.920, but the evaluation proxy marks the page as non-declining. The page has 978 impressions, 20 days since its last update, average position 23.9, CTR 0.00, and engagemen

## Self-check

Before you submit, confirm each line honestly:

- [x] Every section above is filled — markdown thinking AND the code that backs it
- [x] The notebook runs top to bottom with no errors (Runtime → Run all)
- [x] No client names, URLs, or private queries anywhere
- [x] My claims use careful words: observed, measured, directional, decision-support
- [x] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.